In [ ]:
# Colab setup
import sys
if "google.colab" in sys.modules:
    %pip install -q pyedautils ephem statsmodels plotly kaleido ipywidgets

import plotly.io as pio
pio.renderers.default = "notebook_connected"

# Autocorrelation

```{admonition} Goal
:class: tip
Identify dominant periodicities in a time series using autocorrelation. Peaks in the autocorrelation function (ACF) reveal repeating cycles — daily, weekly, or seasonal — which are fundamental to understanding building energy and climate data.
```

```{admonition} Data Basis
:class: note
**Dataset:** `flatTempHum.csv` — Hourly indoor temperature and humidity measurements from four residential flats (A–D).  
**Column used:** `FlatA_Temp`  
**Time range:** October 2018 – September 2020
```

## Data Preparation

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/retomarek/edap/main/edap/sampleData/flatTempHum.csv"
df = pd.read_csv(url, sep=";")
df["time"] = pd.to_datetime(df["time"])
df = df.set_index("time", drop=True)
df.head()

## Visualization

The autocorrelation plot shows how a signal correlates with itself at increasing time lags. The blue shaded area marks the 95% confidence interval — values inside the band are not statistically significant.

In [ ]:
from pyedautils.plots.stat import plot_autocorrelation

fig = plot_autocorrelation(
    df,
    column="FlatA_Temp",
    lags=168,
    title="Autocorrelation — FlatA Temperature (1 week)",
)
fig.show()

```{dropdown} Reading the autocorrelation plot
- **Peak at lag 24** → daily cycle (temperature now resembles temperature 24 hours ago)
- **Peak at lag 168** → weekly cycle (same day last week)
- **Slow overall decay** → strong trend in the data (non-stationary)
- Residential buildings typically show a clear daily cycle but a weak weekly cycle (no weekday/weekend occupancy difference)
```

### Longer Lag Window

Increase the number of lags to look for longer-period cycles.

In [ ]:
fig = plot_autocorrelation(
    df,
    column="FlatA_Temp",
    lags=720,
    title="Autocorrelation — FlatA Temperature (30 days)",
    height=350,
)
fig.show()

```{dropdown} Customization
- Set `lags=24` for a single-day view (useful for sub-hourly data).
- Set `lags=8760` to check for annual cycles (requires at least 2 years of hourly data).
- The confidence band widens at higher lags because fewer data points are available for estimation.
```